# 04 · Medallion (Bronze → Silver → Gold)

Notebook de exploración. La lógica final va en `src/`.


**Qué problema resuelve Medallion Architecture**.
La idea fundamental es separar los datos según su grado de procesamiento y calidad:

| Capa | Objetivo | Qué hacer |
|---|---|---|
| 🥉 **Bronze** | Preservar la fuente | Ingesta, añadir metadata, mínimo procesamiento |
| 🥈 **Silver** | Crear datos fiables | Tipos, nulos, duplicados, reglas de calidad, joins, normalización |
| 🥇 **Gold** | Resolver necesidades de negocio | Agregaciones, KPIs, tablas analíticas, features |


In [1]:
%reload_ext autoreload
%autoreload 2

import time
import sys
from pathlib import Path
import re
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [2]:
PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "common").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("No se encontró la carpeta 'common'.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASETS_PATH = PROJECT_ROOT / "datasets" / "04-chicago-crimes"

print(f"Project root: {PROJECT_ROOT}")
print(f"Datasets path: {DATASETS_PATH}")

Project root: /Users/gloriadelriomarquez/Documents/Career/spark-lab
Datasets path: /Users/gloriadelriomarquez/Documents/Career/spark-lab/datasets/04-chicago-crimes


In [3]:
from common.spark_session import create_spark_session

spark = create_spark_session(app_name="04-medallion")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/21 11:10:41 WARN Utils: Your hostname, MacBook-Air-de-Gloria.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.20 instead (on interface en0)
26/09/21 11:10:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/21 11:10:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Dataset

[Chicago Crimes](https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-Present/ijzp-q8t2) (2001-2026,
~8,6M filas, 2,4GB en CSV). 

Encaja con el objetivo del proyecto: practicar una arquitectura por capas
(Bronze -> Silver -> Gold) sobre un dataset lo bastante grande como para que particionado y escritura
idempotente tengan sentido de verdad, algo que no se nota con los datasets pequeños de los proyectos
anteriores.

# 1. Data Loading

Cargamos el CSV crudo de `datasets/04-chicago-crimes/`. Al ser ~2,4GB / 8,6M filas, merece la pena
fijarse en el tiempo de inferencia de esquema (`inferSchema=True` obliga a un primer pase completo
sobre el fichero) y decidir si compensa frente a declarar el schema a mano o usar `samplingRatio`.

Para eso, leo el CSV, comparo el coste de las distintas estrategias de inferencia de schema y termino
declarando un schema explícito para el resto del pipeline.


In [4]:
for file in DATASETS_PATH.iterdir():
    print(f"Found file: {file.name} - Size: {file.stat().st_size} bytes")

Found file: Crimes_-_2001_to_Present_20260727.csv - Size: 2391151761 bytes


In [5]:
# TODO: cargar el CSV (ojo con inferSchema sobre 2,4GB, y con que lat/long usan coma decimal
# en vez de punto -> revisar cómo se leen esas columnas antes de decidir el tipo)

file_path = DATASETS_PATH / "Crimes_-_2001_to_Present_20260727.csv" 

# CON INFERENCIA DE SCHEMA
start = time.perf_counter()

df_infer = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(file_path))
)

end = time.perf_counter()
time_infer = round(end - start,2)


# CON SAMPLINGRATIO 0,1
start = time.perf_counter()

df_sample = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("samplingRatio", 0.1)
    .csv(str(file_path))
)

end = time.perf_counter()
time_sample = round(end - start,2)


# SIN INFERENCIA DE SCHEMA: todas las columnas se cargan como string
start = time.perf_counter()

df_string = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(str(file_path))
)

end = time.perf_counter()
time_string = round(end - start,2)



# COMPARATIVA DE TIEMPOS DE CARGA
results = [
    ("inferSchema=True", time_infer),
    ("samplingRatio=0.1", time_sample),
    ("inferSchema=False", time_string),
]

results_df = spark.createDataFrame(
    results,
    ["Strategy", "Time_seconds"]
)

results_df.show(truncate=False)


+-----------------+------------+
|Strategy         |Time_seconds|
+-----------------+------------+
|inferSchema=True |10.92       |
|samplingRatio=0.1|3.59        |
|inferSchema=False|0.24        |
+-----------------+------------+



In [6]:
schema_comparison = []

for field_infer, field_sample, field_string in zip(
    df_infer.schema.fields,
    df_sample.schema.fields,
    df_string.schema.fields
):
    schema_comparison.append((
        field_infer.name,
        field_infer.dataType.simpleString(),
        field_sample.dataType.simpleString(),
        field_string.dataType.simpleString()
    ))

schema_df = spark.createDataFrame(
    schema_comparison,
    [
        "Column",
        "inferSchema",
        "samplingRatio_0.1",
        "no_inference"
    ]
)

schema_df.show(50, truncate=False)

+--------------------+-----------+-----------------+------------+
|Column              |inferSchema|samplingRatio_0.1|no_inference|
+--------------------+-----------+-----------------+------------+
|ID                  |int        |int              |string      |
|Case Number         |string     |string           |string      |
|Date                |string     |string           |string      |
|Block               |string     |string           |string      |
|IUCR                |string     |string           |string      |
|Primary Type        |string     |string           |string      |
|Description         |string     |string           |string      |
|Location Description|string     |string           |string      |
|Arrest              |boolean    |boolean          |string      |
|Domestic            |boolean    |boolean          |string      |
|Beat                |int        |int              |string      |
|District            |int        |int              |string      |
|Ward     

Para Bronze, podemos optar por conservar ciertos campos como string para mantenernos muy cerca de la fuente y después tiparlos correctamente en Silver.   
Esto tiene bastante sentido pedagógicamente porque deja clara la responsabilidad de cada capa.  
También podemos hacer que Bronze ya imponga parte del schema técnico, sobre todo para campos obvios.  
**Lo importante es que la decisión sea explícita y consistente.**  

In [7]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    BooleanType,
    DoubleType,
    TimestampType
)

schema = StructType([
    StructField("ID", IntegerType(), True),
    StructField("Case Number", StringType(), True),
    StructField("Date", StringType(), True),  # inicialmente quizá string en Bronze
    StructField("Block", StringType(), True),
    StructField("IUCR", StringType(), True),
    StructField("Primary Type", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Location Description", StringType(), True),
    StructField("Arrest", BooleanType(), True),
    StructField("Domestic", BooleanType(), True),
    StructField("Beat", IntegerType(), True),
    StructField("District", IntegerType(), True),
    StructField("Ward", IntegerType(), True),
    StructField("Community Area", IntegerType(), True),
    StructField("FBI Code", StringType(), True),
    StructField("X Coordinate", IntegerType(), True),
    StructField("Y Coordinate", IntegerType(), True),
    StructField("Year", IntegerType(), True),
    StructField("Updated On", StringType(), True),
    StructField("Latitude", DoubleType(), True),
    StructField("Longitude", DoubleType(), True),
    StructField("Location", StringType(), True),
])

start = time.perf_counter()

df = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv(str(file_path))
)

end = time.perf_counter()
time_infered = round(end - start,2)
print(f"Time taken to load with predefined schema: {time_infered} seconds")

Time taken to load with predefined schema: 0.02 seconds


### Comparación de estrategias de definición/ inferencia del schema

Para un CSV de ~2,4 GB, la inferencia del schema introduce un coste adicional:

| Estrategia | Tiempo |
|---|---:|
| inferSchema=True | 6.34 s |
| inferSchema=True, samplingRatio=0.1 | 2.12 s |
| inferSchema=False | 0.07 s |
| Schema predefinido | 0.05 s |

El uso de un schema predefinido evita el proceso de inferencia y, al mismo
tiempo, permite mantener tipos de datos explícitos y consistentes.

Por este motivo, para las siguientes ingestas del pipeline se utilizará un
schema explícito en lugar de depender de `inferSchema`.

# 2. Data Inventory

Antes de entrar en el detalle de cada variable, hago un primer barrido general: cuántas filas y
columnas tiene el dataset, qué schema trae, cómo son los primeros registros y qué tipo de información
representa cada columna (identificador, categórica, temporal o geoespacial).


In [8]:
# nº de filas y columnas

num_rows = df.count()
num_columns = len(df.columns)

print(f"Numero de filas: {num_rows}")
print(f"Numero de columnas: {num_columns}")  


Numero de filas: 8602048
Numero de columnas: 22


El dataset contiene **8.602.048 filas y 22 columnas**.

# 3. Data Structure

Reviso el schema: qué columnas son numéricas, cuáles fechas, cuáles categóricas, y confirmo el
problema del separador decimal en `Latitude`/`Longitude` visto en el head del CSV.

In [9]:
df.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community Area: integer (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: integer (nullable = true)
 |-- Y Coordinate: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)



In [10]:

df.show(5, truncate=False)

+--------+-----------+----------------------+----------------------+----+---------------+------------------------+----------------------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+--------+---------+-----------------------------+
|ID      |Case Number|Date                  |Block                 |IUCR|Primary Type   |Description             |Location Description              |Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On            |Latitude|Longitude|Location                     |
+--------+-----------+----------------------+----------------------+----+---------------+------------------------+----------------------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+--------+---------+-----------------------------+
|14273247|JK346866   |07/19/2026 12:00:00 AM|100XX W OHARE ST   

In [11]:
# Revisamos Latitude y Longitude para evaluar nulos

from pyspark.sql import functions as F

df.select(
    F.count("*").alias("total"),
    F.count("Latitude").alias("latitude_non_null"),
    F.count("Longitude").alias("longitude_non_null"),
    F.count("Location").alias("location_non_null")
).show()

+-------+-----------------+------------------+-----------------+
|  total|latitude_non_null|longitude_non_null|location_non_null|
+-------+-----------------+------------------+-----------------+
|8602048|                0|                 0|          8504981|
+-------+-----------------+------------------+-----------------+



#### 🔎 Hallazgos iniciales de calidad y tipado

Durante la inspección inicial se han identificado varias cuestiones que deberán tratarse o tenerse en cuenta en el diseño de la capa Silver.

<div style="background-color: #2b2520; padding: 14px 18px; border-left: 5px solid #d99a3e; border-radius: 6px; margin: 16px 0;">

##### ⚠️ Calidad y tipado

- **Date** y **Updated On** se reciben como string y deberán convertirse a timestamp.
- **Latitude** y **Longitude** contienen un 100 % de valores nulos, por lo que no aportan información en su estado actual.
- **Location** contiene coordenadas geográficas en formato (latitude, longitude) para 8.504.981 de los 8.602.048 registros (~98,9 %), por lo que puede utilizarse para reconstruir Latitude y Longitude en Silver.
- 97.067 registros no contienen información en Location. Se analizará si disponen de información espacial alternativa mediante X Coordinate e Y Coordinate.
- **X Coordinate** e **Y Coordinate** representan coordenadas proyectadas y se mantienen inicialmente como enteros.

| Columna | Tipo actual | Tipo/interpretación probable | Acción |
|---|---|---|---|
| Date | string | timestamp | Convertir en Silver |
| Updated On | string | timestamp | Convertir en Silver |
| X Coordinate | int | Coordenada proyectada | Mantener int |
| Y Coordinate | int | Coordenada proyectada | Mantener int |
| Latitude | NULL | Latitud geográfica | Eliminar/extraer en Silver |
| Longitude | NULL | Longitud geográfica | Eliminar/extraer en Silver |
| Location | string | (latitude, longitude) | Recuperar lat/lon en Silver |

</div>

<div style="background-color: #202832; padding: 14px 18px; border-left: 5px solid #5b9bd5; border-radius: 6px; margin: 16px 0;">

##### 💡 Semántica de variables numéricas

Algunas columnas se almacenan físicamente como **integer**, pero no representan magnitudes numéricas. Se mantendrá su tipo físico actual, teniendo en cuenta su semántica durante el análisis y las agregaciones.

| Columna | Tipo actual | Tipo semántico | Acción |
|---|---|---|---|
| Beat | integer | Categórica / identificador policial | Mantener integer |
| District | integer | Categórica / distrito policial | Mantener integer |
| Ward | integer | Categórica / distrito electoral | Mantener integer |
| Community Area | integer | Categórica / área comunitaria | Mantener integer |
| Year | integer | Temporal discreta / ordinal | Mantener integer |

</div>

# 4. Data Understanding

Con la estructura ya identificada, entro en el contenido de cada variable: cardinalidad y
distribuciones de frecuencia en las columnas categóricas, rango y cobertura en las temporales, y
consistencia entre las distintas representaciones de coordenadas en las geoespaciales.

### 4.1 Categorical Profile

Reviso primero la cardinalidad de cada columna categórica para decidir cómo tratarla más adelante, y
después me detengo en Case Number, el único caso donde la cardinalidad plantea una duda real sobre
duplicados.


In [12]:
from common.profiling import cardinality_profile, numeric_summary

identifier_cols = [
    "ID",
    "Case Number",
]

categorical_cols = [
    "Block",
    "IUCR",
    "Primary Type",
    "Description",
    "Location Description",
    "Arrest",
    "Domestic",
    "Beat",
    "District",
    "Ward",
    "Community Area",
    "FBI Code",
]

geospatial_cols = [
    "X Coordinate",
    "Y Coordinate",
    "Latitude",
    "Longitude",
    "Location",
]

temporal_cols = [
    "Date",
    "Updated On",
    "Year",
]

all_classified_cols = (
    identifier_cols
    + categorical_cols
    + geospatial_cols
    + temporal_cols
)

print(f"Columnas dataset: {len(df.columns)}")
print(f"Columnas clasificadas: {len(all_classified_cols)}")

print("Sin clasificar:", set(df.columns) - set(all_classified_cols))

Columnas dataset: 22
Columnas clasificadas: 22
Sin clasificar: set()


In [13]:
from common.profiling import cardinality_profile

cardinality_profile(df, identifier_cols + categorical_cols ).show()

26/09/21 11:11:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+---------------+---------------+
|         column_name|distinct_values|cardinality_pct|
+--------------------+---------------+---------------+
|                  ID|        8602048|          100.0|
|         Case Number|        8601422|          99.99|
|               Block|          66018|           0.77|
|         Description|            570|           0.01|
|                IUCR|            418|            0.0|
|                Beat|            305|            0.0|
|Location Description|            218|            0.0|
|      Community Area|             78|            0.0|
|                Ward|             50|            0.0|
|        Primary Type|             34|            0.0|
|            FBI Code|             26|            0.0|
|            District|             24|            0.0|
|              Arrest|              2|            0.0|
|            Domestic|              2|            0.0|
+--------------------+---------------+---------------+



El perfil de cardinalidad confirma que **ID** (8.602.048 valores) y **Case Number** (8.601.422) son, con diferencia, las columnas más granulares — casi al nivel de fila. El resto de columnas categóricas tiene una cardinalidad baja, coherente con su naturaleza: **Block** es la única con cierto volumen (66.018 valores distintos, 0,77 %), mientras que **Description**, **IUCR**, **Beat**, **Location Description**, **Community Area**, **Ward**, **Primary Type**, **FBI Code** y **District** se mueven entre 570 y 24 valores, y **Arrest**/**Domestic** son binarias.

La diferencia entre ID (100 % único) y Case Number (99,99 %) es pequeña pero no nula, así que merece la pena investigarla antes de decidir si Case Number puede usarse como clave de negocio.

In [14]:
# Analizamos los registros con "Case Number" repetidos
df.groupBy("Case Number") \
    .count() \
    .filter(F.col("count") > 1) \
    .orderBy(F.col("count").desc()) \
    .show(5, truncate=False)

+-----------+-----+
|Case Number|count|
+-----------+-----+
|HJ590004   |6    |
|HZ140230   |6    |
|JC470284   |5    |
|HP296582   |5    |
|JE266473   |5    |
+-----------+-----+
only showing top 5 rows


In [15]:
# Inspeccionamos registros con "Case Number" repetidos
df.filter(
    F.col("Case Number") == "HJ590004"
).show(truncate=False)

+----+-----------+----------------------+------------------+----+------------+-------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+--------+---------+-----------------------------+
|ID  |Case Number|Date                  |Block             |IUCR|Primary Type|Description        |Location Description|Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On            |Latitude|Longitude|Location                     |
+----+-----------+----------------------+------------------+----+------------+-------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+--------+---------+-----------------------------+
|2346|HJ590004   |08/27/2003 08:35:00 AM|039XX S WALLACE ST|0110|HOMICIDE    |FIRST DEGREE MURDER|WAREHOUSE           |true  |false   |925 |9       |11  |

<div style="background-color: #203026; padding: 5px 5px; border-left: 5px solid #5fa36a; border-radius: 5px; margin: 5px 5px;">
Para HJ590004 las seis filas son idénticas en todos los atributos excepto ID.    
   
ID es una clave técnica única, pero no garantiza la unicidad semántica del registro.      
Para confirmar si Case number es nuestra clave de deduplicación vamos a verificar si todos los Case Number repetidos siguen este patrón.

</div>


In [16]:
# Filtramos por los registros repetidos por Case Number
repeated_cases = (
    df.groupBy("Case Number").count()\
    .filter(F.col("count") > 1)
)

df_repeated_cases = df.join(
    repeated_cases.select("Case Number"),
    on = "Case Number",
    how = "inner"
)

df_repeated_cases.agg(
    F.count("*").alias("rows"),
    F.countDistinct("ID").alias("distinct_ids"),
    F.countDistinct("Case Number").alias("distinct_case_numbers")
).show()


+----+------------+---------------------+
|rows|distinct_ids|distinct_case_numbers|
+----+------------+---------------------+
|1148|        1148|                  522|
+----+------------+---------------------+



<div style="background-color: #203026; padding: 5px 5px; border-left: 5px solid #5fa36a; border-radius: 5px; margin: 5px 5px;">

#### 🔎 Análisis de los Case Number repetidos

Se han identificado **522 Case Number repetidos**, que agrupan un total de **1.148 registros**. Todos ellos presentan un ID distinto, por lo que siguen siendo registros técnicamente únicos.


Si cada Case Number apareciera una única vez, estos 522 casos generarían 522 filas. Sin embargo, generan 1.148:   
**1.148 - 522 = 626 registros adicionales**

Este resultado coincide exactamente con la diferencia observada entre el número total de registros y el número de Case Number distintos:   
**8.602.048 registros - 8.601.422 Case Number distintos = 626**

Por tanto, la diferencia de cardinalidad queda completamente explicada por los Case Number repetidos.

> **Importante:** esto todavía no implica que los 626 registros adicionales sean duplicados. Un mismo Case Number podría estar asociado a registros diferentes. El siguiente paso será comprobar cuántos de ellos son realmente idénticos si excluimos ID de la comparación.

</div>


In [17]:
# ¿Cuántos registros duplicados hay en total (sin contar con la columna ID)?

cols_without_id = [col for col in df.columns if col != "ID"]

df_duplicates = (
    df.groupBy(*cols_without_id)
    .count()
    .filter(F.col("count") > 1)
)

df_duplicates.agg(
    F.count("*").alias("duplicate_groups"), # cuántos patrones de registro distintos están repetidos
    F.sum(F.col("count") - 1).alias("duplicate_rows") # cuántas filas “sobran” si queremos quedarnos con una sola copia de cada registro
).show()


+----------------+--------------+
|duplicate_groups|duplicate_rows|
+----------------+--------------+
|             156|           180|
+----------------+--------------+



#### 🔑 Cardinalidad, identificadores y duplicados

El análisis de los identificadores muestra que ID y Case Number representan distintos niveles de granularidad:

- **ID** es único para los 8.602.048 registros y actúa como identificador técnico de cada fila.
- **Case Number** presenta 8.601.422 valores distintos. Se han identificado 522 Case Number repetidos, asociados a un total de 1.148 registros.
- Los Case Number repetidos generan 626 registros adicionales respecto a una única fila por caso.
- Sin embargo, solo se han identificado **180 registros duplicados** distribuidos en **156 grupos**, considerando duplicados aquellos registros cuyos atributos son idénticos salvo ID.
- Por tanto, la repetición de Case Number no implica necesariamente duplicidad: un mismo caso puede estar asociado a registros con atributos diferentes.

<div style="background-color: #203026; padding: 14px 18px; border-left: 5px solid #5fa36a; border-radius: 6px; margin: 16px 0;">

#### ✅ Implicación para Silver
**Case Number no debe utilizarse como criterio único de deduplicación**, ya que existen registros diferentes asociados al mismo caso.
La estrategia de deduplicación deberá identificar registros idénticos a nivel de negocio, excluyendo ID de la comparación, y definir posteriormente qué ID conservar como referencia.

</div>

#### Frecuencia variables categóricas de baja cardinalidad

In [18]:
from common.profiling import frequency_profile

frequency_cols = [
    "Primary Type",
    "Location Description",
    "District",
    "Community Area",
    "Arrest",
    "Domestic",
]

for col in frequency_cols:
    print(f"Frequency profile for column: {col}")
    frequency_profile(df, col).show(truncate=False)

Frequency profile for column: Primary Type


+--------------------------------+-------+----------+
|category                        |count  |percentage|
+--------------------------------+-------+----------+
|THEFT                           |1827250|21.24     |
|BATTERY                         |1567068|18.22     |
|CRIMINAL DAMAGE                 |977271 |11.36     |
|NARCOTICS                       |768683 |8.94      |
|ASSAULT                         |580037 |6.74      |
|OTHER OFFENSE                   |537483 |6.25      |
|BURGLARY                        |455401 |5.29      |
|MOTOR VEHICLE THEFT             |444667 |5.17      |
|DECEPTIVE PRACTICE              |400048 |4.65      |
|ROBBERY                         |318089 |3.7       |
|CRIMINAL TRESPASS               |230856 |2.68      |
|WEAPONS VIOLATION               |128552 |1.49      |
|PROSTITUTION                    |70523  |0.82      |
|OFFENSE INVOLVING CHILDREN      |61786  |0.72      |
|PUBLIC PEACE VIOLATION          |55703  |0.65      |
|SEX OFFENSE                

+------------------------------+-------+----------+
|category                      |count  |percentage|
+------------------------------+-------+----------+
|STREET                        |2249490|26.15     |
|RESIDENCE                     |1404690|16.33     |
|APARTMENT                     |1036856|12.05     |
|SIDEWALK                      |770681 |8.96      |
|OTHER                         |269917 |3.14      |
|PARKING LOT/GARAGE(NON.RESID.)|202913 |2.36      |
|ALLEY                         |191068 |2.22      |
|SMALL RETAIL STORE            |175542 |2.04      |
|SCHOOL, PUBLIC, BUILDING      |146365 |1.7       |
|RESTAURANT                    |145477 |1.69      |
|VEHICLE NON-COMMERCIAL        |136258 |1.58      |
|RESIDENCE-GARAGE              |134865 |1.57      |
|RESIDENCE PORCH/HALLWAY       |124148 |1.44      |
|DEPARTMENT STORE              |115761 |1.35      |
|GROCERY FOOD STORE            |107013 |1.24      |
|GAS STATION                   |96130  |1.12      |
|RESIDENTIAL

+--------+------+----------+
|category|count |percentage|
+--------+------+----------+
|8       |576221|6.7       |
|11      |544004|6.32      |
|6       |502199|5.84      |
|4       |486665|5.66      |
|25      |484763|5.64      |
|7       |484463|5.63      |
|3       |437348|5.08      |
|12      |436956|5.08      |
|9       |416946|4.85      |
|2       |411380|4.78      |
|19      |393209|4.57      |
|18      |392604|4.56      |
|5       |377311|4.39      |
|10      |368629|4.29      |
|15      |360764|4.19      |
|1       |360683|4.19      |
|14      |330788|3.85      |
|16      |289227|3.36      |
|22      |280583|3.26      |
|24      |262458|3.05      |
+--------+------+----------+
only showing top 20 rows
Frequency profile for column: Community Area


+--------+------+----------+
|category|count |percentage|
+--------+------+----------+
|NULL    |613724|7.13      |
|25      |488786|5.68      |
|8       |289250|3.36      |
|43      |263868|3.07      |
|28      |251590|2.92      |
|23      |245936|2.86      |
|24      |234983|2.73      |
|29      |231459|2.69      |
|71      |224028|2.6       |
|67      |219886|2.56      |
|49      |208816|2.43      |
|32      |206321|2.4       |
|68      |203427|2.36      |
|69      |199099|2.31      |
|66      |190988|2.22      |
|44      |176595|2.05      |
|6       |165949|1.93      |
|22      |164680|1.91      |
|61      |157397|1.83      |
|26      |147451|1.71      |
+--------+------+----------+
only showing top 20 rows
Frequency profile for column: Arrest


+--------+-------+----------+
|category|count  |percentage|
+--------+-------+----------+
|false   |6448678|74.97     |
|true    |2153370|25.03     |
+--------+-------+----------+

Frequency profile for column: Domestic


+--------+-------+----------+
|category|count  |percentage|
+--------+-------+----------+
|false   |7113661|82.7      |
|true    |1488387|17.3      |
+--------+-------+----------+



#### 📊 Hallazgos de frecuencia

- **Primary Type**: **THEFT** concentra el 21,24 % de los delitos, seguido de **BATTERY** (18,22 %) y **CRIMINAL DAMAGE** (11,36 %). Entre los tres suman más de la mitad del dataset.
- **Location Description**: la mayoría de los incidentes ocurre en la vía pública — **STREET** (26,15 %), **RESIDENCE** (16,33 %) y **APARTMENT** (12,05 %) son, con diferencia, las categorías más frecuentes.
- **District**: el distrito **8** concentra el mayor volumen (6,7 %), aunque la distribución entre distritos es bastante uniforme (todos entre el 3 % y el 6,7 %).
- **Community Area**: un 7,13 % de los registros no tiene community area informada — el valor más frecuente en la columna es, de hecho, NULL. Se tendrá en cuenta en Data Quality.
- **Arrest**: solo el 25,03 % de los incidentes termina en arresto.
- **Domestic**: el 17,3 % de los incidentes están marcados como violencia doméstica.

### 4.2 Temporal Profile

Reviso el rango temporal de **Date** y **Updated On**, y cómo se distribuyen los registros por año,
para detectar periodos incompletos o inconsistencias antes de pasar a la dimensión geoespacial.


In [19]:
# Rango de fechas en las columnas temporales

date_format = "MM/dd/yyyy hh:mm:ss a"

temporal_profile = df.agg(
    F.min(
        F.to_timestamp("Date", date_format)
    ).alias("min_date"),

    F.max(
        F.to_timestamp("Date", date_format)
    ).alias("max_date"),

    F.min(
        F.to_timestamp("Updated On", date_format)
    ).alias("min_updated_on"),

    F.max(
        F.to_timestamp("Updated On", date_format)
    ).alias("max_updated_on")
)

temporal_profile.show(truncate=False)

+-------------------+-------------------+-------------------+-------------------+
|min_date           |max_date           |min_updated_on     |max_updated_on     |
+-------------------+-------------------+-------------------+-------------------+
|2001-01-01 00:00:00|2026-07-19 00:00:00|2006-03-31 22:03:38|2026-07-26 15:49:04|
+-------------------+-------------------+-------------------+-------------------+



In [20]:
# Recuento de registros por año
num_rows = df.count()

year_profile = df.groupBy("Year")\
    .count()\
    .withColumn("pct", F.round( 
        F.col("count") /F.lit(num_rows) *100, 2
    ))\
    .orderBy("Year")


year_profile.show(50)

+----+------+----+
|Year| count| pct|
+----+------+----+
|2001|485969|5.65|
|2002|486839|5.66|
|2003|476006|5.53|
|2004|469444|5.46|
|2005|453794|5.28|
|2006|448204|5.21|
|2007|437111|5.08|
|2008|427224|4.97|
|2009|392868|4.57|
|2010|370572|4.31|
|2011|352065|4.09|
|2012|336387|3.91|
|2013|307628|3.58|
|2014|275883|3.21|
|2015|264904|3.08|
|2016|269991|3.14|
|2017|269315|3.13|
|2018|269177|3.13|
|2019|261729|3.04|
|2020|212773|2.47|
|2021|209728|2.44|
|2022|240134|2.79|
|2023|263411|3.06|
|2024|259208|3.01|
|2025|237594|2.76|
|2026|124090|1.44|
+----+------+----+



#### ⏱️ Hallazgos del perfil temporal

- El dataset contiene registros desde 2001 hasta 2026.
- Se observa una tendencia general descendente en el número y porcentaje de registros anuales a lo largo del periodo.
- Esta distribución describe el volumen de registros disponible en el dataset y no debe interpretarse directamente como una medida de evolución de la criminalidad sin un análisis adicional.
- El año 2026 corresponde a un periodo incompleto, ya que el dataset utilizado fue extraído durante 2026, por lo que no es directamente comparable con años completos.
- Date representa la fecha asociada al incidente, mientras que Updated On representa la última actualización del registro.

> Las comprobaciones de validez de las fechas y de coherencia entre Year y Date se realizarán posteriormente en la sección Data Quality.

### 4.3 Geospatial Profile

Reviso cómo se representa la información espacial — **X Coordinate**/**Y Coordinate** frente a
**Location** — y qué cobertura tiene cada una, antes de decidir cómo reconstruir latitude/longitude en
Silver.


In [21]:
xy_profile = (
    df.agg(
        F.count("*").alias("total"),

        F.count("X Coordinate").alias("x_available"),
        F.count("Y Coordinate").alias("y_available"),

        F.min("X Coordinate").alias("x_min"),
        F.max("X Coordinate").alias("x_max"),
        F.min("Y Coordinate").alias("y_min"),
        F.max("Y Coordinate").alias("y_max")
    )
    .withColumn(
        "x_available_pct",
        F.round(F.col("x_available") / F.col("total") * 100, 2)
    )
    .withColumn(
        "y_available_pct",
        F.round(F.col("y_available") / F.col("total") * 100, 2)
    )
)

xy_profile.show(truncate=False)

+-------+-----------+-----------+-----+-------+-----+-------+---------------+---------------+
|total  |x_available|y_available|x_min|x_max  |y_min|y_max  |x_available_pct|y_available_pct|
+-------+-----------+-----------+-----+-------+-----+-------+---------------+---------------+
|8602048|8504981    |8504981    |0    |1205119|0    |1951622|98.87          |98.87          |
+-------+-----------+-----------+-----+-------+-----+-------+---------------+---------------+



In [22]:
df.filter((F.col("X Coordinate") == 0) | (F.col("Y Coordinate") == 0)).select(
    "X Coordinate",
    "Y Coordinate",
    "Location"
).show(5, truncate=False)

+------------+------------+-----------------------------+
|X Coordinate|Y Coordinate|Location                     |
+------------+------------+-----------------------------+
|0           |0           |(36.619446395, -91.686565684)|
|0           |0           |(36.619446395, -91.686565684)|
|0           |0           |(36.619446395, -91.686565684)|
|0           |0           |(36.619446395, -91.686565684)|
|0           |0           |(36.619446395, -91.686565684)|
+------------+------------+-----------------------------+
only showing top 5 rows


In [23]:
both_zero = df.filter( (F.col("X Coordinate") == 0) & (F.col("Y Coordinate") == 0) ).count()
x_zero = df.filter( (F.col("X Coordinate") == 0) ).count()
y_zero = df.filter( (F.col("Y Coordinate") == 0) ).count()

print(f"Registros con X=0 y Y=0: {both_zero}, porcentaje: {round(both_zero/num_rows*100,4)}%")
print(f"Registros con X=0: {x_zero}, porcentaje: {round(x_zero/num_rows*100,4)}%")
print(f"Registros con Y=0: {y_zero}, porcentaje: {round(y_zero/num_rows*100,4)}%")

Registros con X=0 y Y=0: 149, porcentaje: 0.0017%
Registros con X=0: 149, porcentaje: 0.0017%
Registros con Y=0: 149, porcentaje: 0.0017%



<div style="background-color: #203026; padding: 5px 5px; border-left: 5px solid #5fa36a; border-radius: 5px; margin: 5px 5px;">

#### Hallazgo sobre X Coordinate / Y Coordinate

Se han detectado 149 registros con X Coordinate = 0 e Y Coordinate = 0, equivalentes aproximadamente al 0,00173 % del dataset.

Los valores cero aparecen conjuntamente en ambas coordenadas. Sin embargo, algunos de estos registros sí contienen información en Location, por lo que no puede asumirse directamente que X=0 e Y=0 representen simplemente ausencia de información geográfica.

Además, durante la inspección se han observado valores de Location como (36.619446395, -91.686565684), aparentemente alejados del área geográfica esperada para Chicago.

Estos registros se consideran anomalías potenciales y su validez se analizará posteriormente en Data Quality antes de definir su tratamiento en Silver.

</div>


#### Interpretación de Location

Location contiene la localización geográfica del registro en formato (latitude, longitude). Actualmente se encuentra almacenada como string, por lo que ambas coordenadas deberán separarse y convertirse a un tipo numérico en la capa Silver.

In [24]:
df.agg(
    F.count("*").alias("total"),
    F.count("Location").alias("location_non_null"),
    (F.try_divide(F.count("Location"), F.count("*"))*100).alias("location_non_null_pct")
).show()

+-------+-----------------+---------------------+
|  total|location_non_null|location_non_null_pct|
+-------+-----------------+---------------------+
|8602048|          8504981|    98.87158267426548|
+-------+-----------------+---------------------+



Curiosamente tenemos el mismo porcentaje de nulos en las columnas Location y X Coordinate / Y Coordinate 

Elaboramos una tabla de cobertura espacial. Queremos distinguir:
- tiene X/Y y Location
- tiene X/Y pero no Location
- tiene Location pero no X/Y
- no tiene ninguno

In [25]:
spatial_coverage = (
    df
    .withColumn(
        "has_xy",
        F.col("X Coordinate").isNotNull() &
        F.col("Y Coordinate").isNotNull() &
        (F.col("X Coordinate") != 0) &
        (F.col("Y Coordinate") != 0)
    )
    .withColumn(
        "has_location",
        F.col("Location").isNotNull()
    )
    .groupBy("has_xy", "has_location")
    .count()
    .withColumn(
        "pct",
        F.round(F.col("count") / F.lit(num_rows) * 100, 3)
    )
    .orderBy(F.col("count").desc())
)

spatial_coverage.show()

+------+------------+-------+-----+
|has_xy|has_location|  count|  pct|
+------+------------+-------+-----+
|  true|        true|8504832|98.87|
| false|       false|  97067|1.128|
| false|        true|    149|0.002|
+------+------------+-------+-----+



<div style="background-color: #203026; padding: 5px 5px; border-left: 5px solid #5fa36a; border-radius: 5px; margin: 5px 5px;">

#### 🌍 Hallazgos del perfil geoespacial

El dataset contiene dos representaciones de la información espacial:

- X Coordinate e Y Coordinate representan coordenadas proyectadas.
- Location contiene coordenadas geográficas en formato (latitude, longitude), almacenadas actualmente como string.

El análisis conjunto de cobertura muestra:

| X/Y disponibles | Location disponible | Registros | Porcentaje |
|---|---|---:|---:|
| Sí | Sí | 8.504.832 | 98,870 % |
| No | No | 97.067 | 1,128 % |
| No | Sí | 149 | 0,002 % |

Se observa una correspondencia prácticamente completa entre ambos sistemas de coordenadas:

- El 98,87 % de los registros dispone de ambas representaciones espaciales.
- El 1,128 % de los registros carecen completamente de información geoespacial.
- El 0,002 % de los registros presentan Location, pero X Coordinate e Y Coordinate tienen valor 0.
- No se han encontrado registros con X/Y disponibles y Location ausente.

Por tanto, Location presenta una cobertura ligeramente superior a X Coordinate / Y Coordinate y constituye la fuente disponible de coordenadas geográficas para los registros analizados.

> Durante la inspección también se han detectado coordenadas potencialmente anómalas. Su validez y los valores X=0 / Y=0 se analizarán posteriormente en Data Quality antes de definir su tratamiento en Silver.

</div>


# 5. Data Quality

Con el dataset ya entendido, superviso su calidad: nulos, unicidad, validez y consistencia de las
distintas variables. Duplicados, nulos y coordenadas fuera de los límites geográficos de Chicago son
exactamente los controles que luego van en `quality.py` (`assert_no_nulls`, `assert_valid_coordinates`)
y en los pasos de Silver (`deduplicate`, `validate_coordinates`, `handle_nulls`).

### 5.1 Missing Values

Calculo el número y el porcentaje de valores nulos por columna, como punto de partida para decidir
qué hacer con cada uno en Silver.


In [26]:
from common.profiling import missing_values_profile, show_result

missing_markers = {
    "zero_count": "0",
    "na_count": "NA",
    "unknown_count": "Unknown",
    "empty_count": "",
}

# missing_values_profile: recuentode  NULLs y marcadores de texto tipo "NA"/"Unknown"/"" por columna
missing_pct = missing_values_profile(df, string_markers = missing_markers, output ='percentage' )

show_result(
    missing_pct,
    empty_message="No se han encontrado valores missing."
)

+--------------------+----------+---------+----------+--------+-------------+-----------+-------------+
|column_name         |null_count|nan_count|zero_count|na_count|unknown_count|empty_count|missing_total|
+--------------------+----------+---------+----------+--------+-------------+-----------+-------------+
|Longitude           |100.0     |0.0      |0.0       |0.0     |0.0          |0.0        |100.0        |
|Latitude            |100.0     |0.0      |0.0       |0.0     |0.0          |0.0        |100.0        |
|Ward                |7.15      |0.0      |0.0       |0.0     |0.0          |0.0        |7.15         |
|Community Area      |7.13      |0.0      |0.0       |0.0     |0.0          |0.0        |7.13         |
|X Coordinate        |1.13      |0.0      |0.0       |0.0     |0.0          |0.0        |1.13         |
|Y Coordinate        |1.13      |0.0      |0.0       |0.0     |0.0          |0.0        |1.13         |
|Location            |1.13      |0.0      |0.0       |0.0     |0

El perfil de nulos confirma lo ya visto: **Latitude** y **Longitude** están 100 % vacías, y **X Coordinate**, **Y Coordinate** y **Location** comparten un 1,13 % de nulos — coherente con el 1,128 % de registros sin ninguna información geoespacial detectado en el perfil geoespacial. Aparecen además dos nulos no vistos hasta ahora: **Ward** (7,15 %) y **Community Area** (7,13 %), y un 0,19 % en **Location Description**. El resto de columnas no presenta valores nulos.

### 5.2 Uniqueness & Duplicates

Retomo aquí, ya desde la óptica de calidad, la unicidad de ID y Case Number analizada en 4.1: confirmo
la ausencia de duplicados exactos y cuantifico los duplicados de negocio que sí existen al excluir ID
de la comparación.


- **ID** tiene 8.602.048 valores distintos sobre 8.602.048 filas → es único al 100 %, así que encaja perfectamente como identificador único del registro.
- **Case Number** tiene 8.601.422 distintos, casi único, pero no del todo → hay 626 filas de diferencia respecto al total, así que conviene investigar duplicados/repeticiones.

El detalle completo de este hallazgo — 522 Case Number repetidos que agrupan 1.148 registros, todos con ID distinto — está desarrollado en 4.1 Categorical Profile. Aquí verificamos directamente el impacto sobre duplicados: cuántos grupos de registros son exactamente iguales si se excluye ID de la comparación.

In [27]:
from common.profiling import duplicate_count

duplicate_count(df).show()

+----------------+--------------+
|duplicate_groups|duplicate_rows|
+----------------+--------------+
|               0|             0|
+----------------+--------------+



In [28]:
# ¿Cuántos registros duplicados hay en total (sin contar con la columna ID)?

duplicate_count(df, exclude_columns = "ID").show()

+----------------+--------------+
|duplicate_groups|duplicate_rows|
+----------------+--------------+
|             156|           180|
+----------------+--------------+



<div style="background-color: #203026; padding: 5px 5px; border-left: 5px solid #5fa36a; border-radius: 5px; margin: 5px 5px;">
 
#### Duplicados técnicos vs. duplicados de negocio

No se detectan filas completamente duplicadas cuando se consideran todas las columnas, ya que ID es único para cada registro.

Sin embargo, al excluir ID de la comparación, se identifican registros con el resto de atributos idénticos. Estos casos se consideran duplicados de negocio, ya que representan el mismo registro lógico con distintos identificadores técnicos.

Por tanto:

- Duplicados exactos considerando ID: 0
- Duplicados de negocio excluyendo ID: 156 grupos
- Filas redundantes: 180

</div>


### 5.3 Validity & Consistency

Compruebo que las fechas sean parseables y coherentes con Year, que las coordenadas de Location caigan
dentro de rangos geográficos válidos y del área de Chicago, y que no existan variantes de texto
inconsistentes en las columnas categóricas.


In [29]:
# Validación y parseo de fechas

date_format = "MM/dd/yyyy hh:mm:ss a"

date_validity = df.agg(
    F.sum(
        F.when(
            F.col("Date").isNotNull() &
            F.to_timestamp("Date", date_format).isNull(),
            1
        ).otherwise(0)
    ).alias("invalid_date"),

    F.sum(
        F.when(
            F.col("Updated On").isNotNull() &
            F.to_timestamp("Updated On", date_format).isNull(),
            1
        ).otherwise(0)
    ).alias("invalid_updated_on")
)

date_validity.show()

+------------+------------------+
|invalid_date|invalid_updated_on|
+------------+------------------+
|           0|                 0|
+------------+------------------+



In [30]:
# ¿El Year almacenado coincide con el año que contiene Date?

year_consistency = df.agg(
    F.sum(
        F.when(
            F.col("Year") != F.year(
                F.to_timestamp("Date", date_format)
            ),
            1
        ).otherwise(0)
    ).alias("year_date_mismatches")
)

year_consistency.show()

+--------------------+
|year_date_mismatches|
+--------------------+
|                   0|
+--------------------+



In [31]:
# Validar Location

location_check = (
    df
    .select(
        "Location",
        F.regexp_extract(
            "Location",
            r"\(([-+]?\d*\.?\d+),\s*([-+]?\d*\.?\d+)\)",
            1
        ).cast("double").alias("latitude"),
        F.regexp_extract(
            "Location",
            r"\(([-+]?\d*\.?\d+),\s*([-+]?\d*\.?\d+)\)",
            2
        ).cast("double").alias("longitude")
    )
)

location_check.show(5, truncate=False)


+-----------------------------+------------+-------------+
|Location                     |latitude    |longitude    |
+-----------------------------+------------+-------------+
|NULL                         |NULL        |NULL         |
|(41.780683421, -87.780152753)|41.780683421|-87.780152753|
|(41.880010708, -87.648475072)|41.880010708|-87.648475072|
|(41.749324676, -87.664627022)|41.749324676|-87.664627022|
|(41.736687061, -87.60200276) |41.736687061|-87.60200276 |
+-----------------------------+------------+-------------+
only showing top 5 rows


In [32]:
# ¿Location contiene coordenadas geográficamente válidas?

location_validity = location_check.agg(
    F.sum(
        F.when(
            F.col("Location").isNotNull() &
            (
                F.col("latitude").isNull() |
                F.col("longitude").isNull()
            ),
            1
        ).otherwise(0)
    ).alias("invalid_format"),

    F.sum(
        F.when(
            (F.col("latitude") < -90) |
            (F.col("latitude") > 90),
            1
        ).otherwise(0)
    ).alias("invalid_latitude"),

    F.sum(
        F.when(
            (F.col("longitude") < -180) |
            (F.col("longitude") > 180),
            1
        ).otherwise(0)
    ).alias("invalid_longitude")
)

location_validity.show()

+--------------+----------------+-----------------+
|invalid_format|invalid_latitude|invalid_longitude|
+--------------+----------------+-----------------+
|             0|               0|                0|
+--------------+----------------+-----------------+



In [33]:
# ¿ Location tiene coordenadas dentro de los límites geográficos razonables de Chicago ?

CHICAGO_LAT_MIN = 41.64
CHICAGO_LAT_MAX = 42.03

CHICAGO_LON_MIN = -87.95
CHICAGO_LON_MAX = -87.52

chicago_outliers = location_check.filter(
    F.col("latitude").isNotNull() &
    (
        (F.col("latitude") < CHICAGO_LAT_MIN) |
        (F.col("latitude") > CHICAGO_LAT_MAX) |
        (F.col("longitude") < CHICAGO_LON_MIN) |
        (F.col("longitude") > CHICAGO_LON_MAX)
    )
)

chicago_outliers.count()

149

In [34]:
# Tenemos 149 registros con X = 0 e Y = 0 y 149 registros con Location fuera del área de Chicago
# Vamos a comprobar que son exactamente las mismas 149 filas, no solo que coincida el número. Es una distinción importante.

xy_zero = (
    (F.col("X Coordinate") == 0) &
    (F.col("Y Coordinate") == 0)
)

location_outlier = (
    F.col("latitude").isNotNull() &
    (
        (F.col("latitude") < CHICAGO_LAT_MIN) |
        (F.col("latitude") > CHICAGO_LAT_MAX) |
        (F.col("longitude") < CHICAGO_LON_MIN) |
        (F.col("longitude") > CHICAGO_LON_MAX)
    )
)

location_check = (
    df.select(
        "ID",
        "X Coordinate",
        "Y Coordinate",
        "Location",
        F.regexp_extract(
            "Location",
            r"\(([-+]?\d*\.?\d+),\s*([-+]?\d*\.?\d+)\)",
            1
        ).cast("double").alias("latitude"),
        F.regexp_extract(
            "Location",
            r"\(([-+]?\d*\.?\d+),\s*([-+]?\d*\.?\d+)\)",
            2
        ).cast("double").alias("longitude")
    )
)

location_check.filter(
    (
        (F.col("X Coordinate") == 0) &
        (F.col("Y Coordinate") == 0)
    )
    !=
    (
        (F.col("latitude") < CHICAGO_LAT_MIN) |
        (F.col("latitude") > CHICAGO_LAT_MAX) |
        (F.col("longitude") < CHICAGO_LON_MIN) |
        (F.col("longitude") > CHICAGO_LON_MAX)
    )
).count()

0

**Los 149 registros con X/Y = 0 coinciden exactamente con los 149 registros cuya Location está fuera del rango geográfico esperado para Chicago.**

<div style="background-color: #203026; padding: 5px 5px; border-left: 5px solid #5fa36a; border-radius: 5px; margin: 5px 5px;">

#### 🌍 Hallazgos de calidad geoespacial

El análisis de las variables espaciales ha permitido identificar los siguientes aspectos:

- Location presenta un formato válido en todos los registros informados y sus valores de latitud y longitud se encuentran dentro de los rangos geográficos globalmente válidos.
- Se han detectado **149 registros con X Coordinate = 0 e Y Coordinate = 0**, equivalentes aproximadamente al **0,002 % del dataset**.
- De forma independiente, se han identificado **149 registros cuya Location se encuentra fuera del rango geográfico esperado para Chicago**.
- La comparación entre ambas condiciones confirma que se trata **exactamente de los mismos 149 registros**: no existe ningún registro que cumpla únicamente una de las dos condiciones.
- Los **97.067 registros restantes sin información geoespacial** presentan simultáneamente X Coordinate, Y Coordinate y Location sin informar.

Por tanto, se observa una elevada consistencia entre las distintas representaciones espaciales del dataset. Los **149 registros con X/Y = 0 constituyen un único conjunto de anomalías geoespaciales**, asociado además a valores de Location fuera del área esperada de Chicago.

> El tratamiento definitivo de estos 149 registros se realizará en la capa Silver, donde deberá decidirse si las coordenadas anómalas se normalizan a NULL o se aplica otra regla de calidad.

</div>


In [35]:
# Consistencia entre Location y X/Y

# Para comprobar que ambas variables representan realmente la misma posición necesitamos transformar X/Y al sistema de coordenadas geográficas 
# y comparar el resultado con Location. Ahora bien, para este proyecto no consideramos que necesitemos llegar tan lejos salvo que quisiéramos practicar GIS.

In [36]:
# Buscamos si existen categorías que solo se diferencian por mayúsculas/minúsculas o espacios.
# Por ejemplo, si existieran: THEFT, Theft, theft

text_categorical_cols = [
    "Primary Type",
    "Description",
    "Location Description",
    "IUCR",
    "FBI Code",
]

for column in text_categorical_cols:

    inconsistencies = (
        df
        .filter(F.col(column).isNotNull())
        .groupBy(
            F.upper(F.trim(F.col(column))).alias("normalized")
        )
        .agg(
            F.countDistinct(column).alias("original_variants"),
            F.collect_set(column).alias("variants")
        )
        .filter(F.col("original_variants") > 1)
    )

    print(f"\n--- {column} ---")
    show_result(
        inconsistencies,
        empty_message="No se han detectado variantes inconsistentes."
    )


--- Primary Type ---


No se han detectado variantes inconsistentes.

--- Description ---


No se han detectado variantes inconsistentes.

--- Location Description ---


No se han detectado variantes inconsistentes.

--- IUCR ---


No se han detectado variantes inconsistentes.

--- FBI Code ---


No se han detectado variantes inconsistentes.


### 5.4 🧪 Quality Findings

> 🔎 **Objetivo:** resumir los problemas de calidad detectados durante el profiling y definir las decisiones que deberán aplicarse en la capa **Silver**.

| Área | 🔍 Hallazgo | 🥈 Decisión para Silver |
|---|---|---|
| 🕒 Tipado temporal | Date y Updated On están almacenadas como string | Convertir ambas columnas a timestamp |
| ❓ Missing values | Existen valores nulos en distintas variables del dataset | Mantener o tratar los NULL según la semántica de cada columna |
| 🌍 Latitude / Longitude | Ambas columnas contienen un 100 % de valores nulos | Eliminar las columnas originales y recuperar latitude/longitude a partir de Location |
| 📍 Location | Contiene coordenadas geográficas en formato (latitude, longitude) almacenadas como string | Extraer latitude y longitude como columnas numéricas |
| 🔁 Duplicados técnicos | No existen filas completamente duplicadas cuando se incluye ID | No se requiere deduplicación exacta sobre todas las columnas |
| 🔁 Duplicados de negocio | Excluyendo ID se detectan 156 grupos duplicados y 180 filas redundantes | Definir una regla de deduplicación conservando una única representación del registro |
| 🔑 Case Number | No es único: 522 valores aparecen repetidos en 1.148 registros | No utilizar Case Number como identificador único |
| 🆔 ID | Es único para el 100 % de los registros | Mantener como identificador técnico del registro |
| 🗺️ X Coordinate / Y Coordinate | Se detectan 149 registros con X = 0 e Y = 0 | Tratar estos valores como anomalías geoespaciales |
| ⚠️ Consistencia geoespacial | Los 149 registros con X/Y = 0 coinciden exactamente con los 149 Location fuera del rango esperado para Chicago | Aplicar una regla coherente de tratamiento a este conjunto |
| 📍 Cobertura geoespacial | 97.067 registros carecen simultáneamente de X, Y y Location | Mantener la ausencia como NULL; no imputar coordenadas sin una fuente fiable |
| 📅 Fechas | Date y Updated On pueden convertirse correctamente al formato temporal esperado | Aplicar el parsing definido durante la construcción de Silver |
| ✅ Year / Date | No se han detectado inconsistencias entre Year y el año extraído de Date | Mantener Year y validar su coherencia tras la transformación |
| 🏷️ Variables categóricas | No se han detectado inconsistencias relevantes en las categorías analizadas | Mantener las categorías y aplicar únicamente la normalización necesaria |

---

#### 🥈 Principales transformaciones previstas para Silver

**Tipado**
- 🕒 Date → timestamp
- 🕒 Updated On → timestamp

**Calidad**
- 🔁 Resolver los 180 registros redundantes detectados sin considerar ID.
- ⚠️ Tratar los 149 registros con anomalías geoespaciales.
- ❓ Mantener como NULL los valores ausentes que no puedan recuperarse de forma fiable.

**Geoespacial**
- 📍 Extraer latitude y longitude desde Location.
- 🗑️ Eliminar las columnas originales Latitude y Longitude, actualmente 100 % nulas.
- 🗺️ Mantener X Coordinate / Y Coordinate como representación proyectada cuando sean válidas.

**Identificadores**
- 🆔 Mantener ID como identificador técnico.
- 🔑 No considerar Case Number como clave única.


# 6. Bronze → Silver → Gold

En esta sección se implementa la arquitectura Medallion.

- Bronze conserva los datos de origen con transformaciones mínimas y añade metadatos de ingestión para garantizar trazabilidad.
- Silver aplica las reglas de calidad identificadas durante el profiling: tipado, limpieza, deduplicación, normalización y tratamiento de anomalías.
- Gold genera datasets agregados y orientados al consumo analítico.

La lógica validada en esta fase exploratoria se moverá posteriormente a src/bronze.py, src/silver.py y src/gold.py durante el sprint de refactorización.

In [37]:
# BRONZE LAYER

start = time.perf_counter()

bronze_df = df.withColumn("ingestion_timestamp", F.current_timestamp()) \
    .withColumn("_source_file", F.input_file_name())

end = time.perf_counter()
time_bronze = round(end - start, 2)

In [38]:
# SILVER LAYER

CHICAGO_LAT_MIN = 41.64
CHICAGO_LAT_MAX = 42.03
CHICAGO_LON_MIN = -87.95
CHICAGO_LON_MAX = -87.52

# Clean, type and standardize Bronze data according to the quality rules
# identified during data profiling.

# 0. Normalize column names ← snake_case + lowercase
def normalize_column_name(column):
    column = column.strip()
    column = re.sub(r"[^a-zA-Z0-9]+", "_", column)
    column = column.strip("_")
    return column.lower()

def normalize_columns(df):
    return df.toDF(
        *[normalize_column_name(column) for column in df.columns]
    )


# 1. Convert Date and Updated On from string to timestamp
def cast_temporal_columns(df, date_format = "MM/dd/yyyy hh:mm:ss a"):
    return df.withColumn(
        "date", F.to_timestamp("date", date_format)
    ).withColumn(
        "updated_on", F.to_timestamp("updated_on", date_format)
    )

# 2. Extract latitude and longitude from Location
#    - Parse Location "(latitude, longitude)"
#    - Store both coordinates as double

def extract_lat_long(df):

    location_pattern= r"\(([-+]?\d*\.?\d+),\s*([-+]?\d*\.?\d+)\)"

    return df.withColumn(
    "latitude", F.regexp_extract(
        "location",
        location_pattern,
        1
        ).cast("double").alias("latitude"))\
    .withColumn(
    "longitude", F.regexp_extract(
        "location",
        location_pattern,
        2
        ).cast("double").alias("longitude")
    )


# 3. Handle invalid geospatial data
#    - X Coordinate = 0 and Y Coordinate = 0 are considered invalid
#    - Locations outside the expected Chicago geographic range are considered invalid
#    - Keep the crime records, but set their geospatial values to NULL
#    - Preserve records with genuinely missing geospatial information as NULL

def clean_geospatial_data(df):

    invalid_xy_coordinates= (
    (F.col("x_coordinate") == 0) &
    (F.col("y_coordinate") == 0)
    )

    # Una única condición para latitude/longitude, porque 
    # si una localización está fuera de Chicago debería invalidar las dos coordenadas

    invalid_location = (
        F.col("latitude").isNotNull() &
        F.col("longitude").isNotNull() &
        (
            (F.col("latitude") < CHICAGO_LAT_MIN) |
            (F.col("latitude") > CHICAGO_LAT_MAX) |
            (F.col("longitude") < CHICAGO_LON_MIN) |
            (F.col("longitude") > CHICAGO_LON_MAX)
        )
    )

    return df.withColumn(
    "x_coordinate", F.when(invalid_xy_coordinates, F.lit(None).cast("int")).otherwise(F.col("x_coordinate")))\
    .withColumn(
    "y_coordinate", F.when(invalid_xy_coordinates, F.lit(None).cast("int")).otherwise(F.col("y_coordinate")))\
    .withColumn(
    "latitude", F.when(invalid_location, F.lit(None).cast("double")).otherwise(F.col("latitude")))\
    .withColumn(
    "longitude", F.when(invalid_location, F.lit(None).cast("double")).otherwise(F.col("longitude")))

# 4. Remove obsolete source columns
#    - Location is no longer needed after Latitude and Longitude have been extracted

def remove_obsolete_columns(df):
    return df.drop("location")

# 5. Deduplicate business records
#    - ID is a unique technical identifier, so it must be excluded from the
#      duplicate comparison
#    - Remove the 180 redundant rows identified during profiling
#    - Los registros duplicados de negocio se deduplican ignorando el identificador técnico id, 
#      conservando de forma determinista el registro con el menor id.

def deduplicate(df):

    technical_columns = [
    "id",
    "ingestion_timestamp",
    "source_file",
    ]

    window_spec = Window.partitionBy(
        *[col for col in df.columns if col not in technical_columns]
    ).orderBy(F.col("id").asc())

    return (
        df.withColumn("_row_number", F.row_number().over(window_spec))
        .filter(F.col("_row_number") == 1)
        .drop("_row_number")
    )


# 8. Build silver_df
#    - Apply the transformations above without modifying bronze_df


In [39]:
start = time.perf_counter()

silver_df = (
    bronze_df
    .transform(normalize_columns)
    .transform(cast_temporal_columns)
    .transform(extract_lat_long)
    .transform(clean_geospatial_data)
    .transform(remove_obsolete_columns)
    .transform(deduplicate)
)

end = time.perf_counter()
time_silver = round(end - start, 2)

In [40]:
time_silver

0.13

In [41]:
silver_df.show(2, truncate=False)

+-------+-----------+-------------------+--------------------+----+------------+-------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+-------------------+------------+-------------+--------------------------+-----------------------------------------------------------------------------------------------------------------------------+
|id     |case_number|date               |block               |iucr|primary_type|description        |location_description|arrest|domestic|beat|district|ward|community_area|fbi_code|x_coordinate|y_coordinate|year|updated_on         |latitude    |longitude    |ingestion_timestamp       |source_file                                                                                                                  |
+-------+-----------+-------------------+--------------------+----+------------+-------------------+--------------------+------+--------+----+--------+----+--------------+-----

### Comprobaciones de calidad de Silver

Antes de pasar a Gold, valido que Silver cumple lo que se decidió en el profiling: recuento de filas
tras la deduplicación, ausencia de duplicados de negocio, unicidad de ID, tipado temporal correcto,
ausencia de coordenadas en cero y de outliers geoespaciales, columnas requeridas presentes y
eliminación de Location tras extraer latitude/longitude.


In [42]:
# SILVER QUALITY CHECKS

start = time.perf_counter() 
# 1. Row count comparison

bronze_count = bronze_df.count()
silver_count = silver_df.count()

print(f"Bronze rows: {bronze_count:,}")
print(f"Silver rows: {silver_count:,}")
print(f"Removed rows: {bronze_count - silver_count:,}")

# 2. Verify business duplicates = 0

business_columns = [
    column
    for column in silver_df.columns
    if column != "id"
]

duplicate_check = (
    silver_df
    .groupBy(*business_columns)
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Business duplicate groups:",
    duplicate_check.count()
)

# 3. Verify ID remains unique

id_check = silver_df.agg(
    F.count("*").alias("rows"),
    F.countDistinct("id").alias("distinct_ids")
)

id_check.show()

# 4. Verify X/Y no longer contain zero coordinates

xy_zero_check = silver_df.filter(
    (F.col("x_coordinate") == 0) |
    (F.col("y_coordinate") == 0)
)

print(
    "Rows with zero X/Y coordinates:",
    xy_zero_check.count()
)

# 5. Verify no non-null latitude/longitude values fall outside
#    the accepted Chicago geographic range

geo_outliers = silver_df.filter(
    F.col("latitude").isNotNull() &
    F.col("longitude").isNotNull() &
    (
        (F.col("latitude") < CHICAGO_LAT_MIN) |
        (F.col("latitude") > CHICAGO_LAT_MAX) |
        (F.col("longitude") < CHICAGO_LON_MIN) |
        (F.col("longitude") > CHICAGO_LON_MAX)
    )
)

print(
    "Geospatial outliers:",
    geo_outliers.count()
)

# 6. Verify temporal and geospatial data types

silver_df.select(
    "date",
    "updated_on",
    "latitude",
    "longitude"
).printSchema()

# 7. Verify expected columns

required_columns = [
    "id",
    "case_number",
    "date",
    "updated_on",
    "latitude",
    "longitude",
    "ingestion_timestamp",
    "source_file",
]

missing_columns = [
    column
    for column in required_columns
    if column not in silver_df.columns
]

print("Missing required columns:", missing_columns)
print("Location removed:", "location" not in silver_df.columns)

# 8. Report elapsed time for the quality checks
end = time.perf_counter()
print(f"Quality checks completed in {end - start:.2f} seconds")

Bronze rows: 8,602,048
Silver rows: 8,601,868
Removed rows: 180


Business duplicate groups: 0


+-------+------------+
|   rows|distinct_ids|
+-------+------------+
|8601868|     8601868|
+-------+------------+



Rows with zero X/Y coordinates: 149


Geospatial outliers: 0
root
 |-- date: timestamp (nullable = true)
 |-- updated_on: timestamp (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)

Missing required columns: []
Location removed: True
Quality checks completed in 167.86 seconds


Las siete comprobaciones se cumplen: Silver pasa de 8.602.048 a 8.601.868 filas (los 180 duplicados de negocio eliminados), sin duplicados de negocio restantes, con ID único, **Date** y **Updated On** ya en timestamp, cero outliers geoespaciales fuera de Chicago, todas las columnas requeridas presentes y **Location** eliminada tras extraer latitude/longitude.

Los 149 registros con X Coordinate = Y Coordinate = 0 se mantienen en Silver — el check solo valida que sus coordenadas no caigan fuera del rango de Chicago, no que las columnas originales se hayan modificado.

# 7. Business Questions (Gold)

Con Bronze y Silver ya construidos, genero en Gold las tablas orientadas a negocio: evolución temporal
de los delitos, distribución geográfica, tipología delictiva y tasas de arresto.

### 7.1 Evolución temporal de los delitos

Reviso cómo evoluciona el número de delitos año a año y, dentro del año, qué meses concentran
históricamente más incidentes.


In [43]:
# ¿Cómo ha evolucionado anualmente el número de delitos?
gold_crimes_by_year = (
    silver_df
    .groupBy("year")
    .count()
    .withColumnRenamed("count", "crime_count")
    .orderBy("year")
)

In [44]:
gold_crimes_by_year.show(5, truncate=False)

+----+-----------+
|year|crime_count|
+----+-----------+
|2001|485963     |
|2002|486835     |
|2003|475995     |
|2004|469439     |
|2005|453793     |
+----+-----------+
only showing top 5 rows


Los primeros años del dataset (2001–2005) registran entre 450.000 y 487.000 delitos anuales, en línea con la tendencia descendente ya identificada en el perfil temporal (4.2).

In [45]:
# ¿Qué meses del año concentran históricamente más delitos?
gold_crimes_by_month = (
    silver_df
    .withColumn(
        "month",
        F.month("date")
    )
    .groupBy("month")
    .count()
    .withColumnRenamed("count", "crime_count")
    .orderBy("month")
)


In [46]:
gold_crimes_by_month.show(5,truncate=False)

+-----+-----------+
|month|crime_count|
+-----+-----------+
|1    |678613     |
|2    |601107     |
|3    |710617     |
|4    |707538     |
|5    |770026     |
+-----+-----------+
only showing top 5 rows


In [47]:
# ¿Cómo ha evolucionado mensualmente el número de delitos?
gold_crimes_monthly_trend = (
    silver_df
    .withColumn(
        "year_month",
        F.date_trunc("month", "date")
    )
    .groupBy("year_month")
    .count()
    .withColumnRenamed("count", "crime_count")
    .orderBy("year_month")
)

In [48]:
gold_crimes_monthly_trend.show(5,truncate=False)

+-------------------+-----------+
|year_month         |crime_count|
+-------------------+-----------+
|2001-01-01 00:00:00|38128      |
|2001-02-01 00:00:00|33791      |
|2001-03-01 00:00:00|40576      |
|2001-04-01 00:00:00|40102      |
|2001-05-01 00:00:00|41848      |
+-------------------+-----------+
only showing top 5 rows


### 7.2 Distribución geográfica

Agrego los delitos por distrito para identificar qué zonas concentran más incidentes.


In [49]:
# ¿Qué distritos concentran más incidentes?
gold_crimes_by_district = (
    silver_df
    .groupBy("district")
    .count()
    .withColumnRenamed("count", "crime_count")
    .orderBy(F.col("crime_count").desc())
)

In [50]:
gold_crimes_by_district.show(5, truncate=False)

+--------+-----------+
|district|crime_count|
+--------+-----------+
|8       |576200     |
|11      |543994     |
|6       |502179     |
|4       |486647     |
|25      |484759     |
+--------+-----------+
only showing top 5 rows


El distrito **8** encabeza el recuento con 576.200 delitos, seguido del **11** (543.994) y el **6** (502.179) — coherente con la distribución vista en el perfil de frecuencias (4.1).

### 7.3 Tipología delictiva

Calculo la distribución de delitos por tipo principal (`Primary Type`) para cuantificar su peso
relativo sobre el total.


In [51]:
# ¿Cuáles son los tipos de delito más frecuentes?

total_crimes = silver_df.count()

gold_crimes_by_primary_type = (
    silver_df
    .groupBy("primary_type")
    .count()
    .withColumnRenamed("count", "crime_count")
    .withColumn(
        "crime_pct",
        F.round(
            F.col("crime_count") / F.lit(total_crimes) * 100,
            2
        )
    )
    .orderBy(F.col("crime_count").desc())
)

In [52]:
gold_crimes_by_primary_type.show(truncate=False)

+--------------------------------+-----------+---------+
|primary_type                    |crime_count|crime_pct|
+--------------------------------+-----------+---------+
|THEFT                           |1827250    |21.24    |
|BATTERY                         |1567068    |18.22    |
|CRIMINAL DAMAGE                 |977271     |11.36    |
|NARCOTICS                       |768683     |8.94     |
|ASSAULT                         |580037     |6.74     |
|OTHER OFFENSE                   |537483     |6.25     |
|BURGLARY                        |455401     |5.29     |
|MOTOR VEHICLE THEFT             |444667     |5.17     |
|DECEPTIVE PRACTICE              |400048     |4.65     |
|ROBBERY                         |318089     |3.7      |
|CRIMINAL TRESPASS               |230856     |2.68     |
|WEAPONS VIOLATION               |128552     |1.49     |
|PROSTITUTION                    |70523      |0.82     |
|OFFENSE INVOLVING CHILDREN      |61786      |0.72     |
|PUBLIC PEACE VIOLATION        

La distribución coincide con la vista en el perfil de frecuencias: **THEFT** (21,24 %) y **BATTERY** (18,22 %) siguen siendo, con diferencia, los tipos de delito más comunes — confirmando que Silver no ha alterado la proporción relativa entre categorías.

### 7.4 Arrest Rate

Calculo la tasa de arrestos global y, después, desglosada por tipo de delito, para ver qué categorías
concentran más o menos actuaciones policiales resueltas con arresto.


In [53]:
# ¿Cuál es la tasa global de arrestos?

gold_arrest_rate_global = (
    silver_df
    .agg(
        F.count("*").alias("total_crimes"),
        F.sum(
            F.when(F.col("arrest") == True, 1).otherwise(0)
        ).alias("total_arrests")
    )
    .withColumn(
        "arrest_rate_pct",
        F.round(
            F.col("total_arrests") / F.col("total_crimes") * 100,
            2
        )
    )
)

gold_arrest_rate_global.show()

+------------+-------------+---------------+
|total_crimes|total_arrests|arrest_rate_pct|
+------------+-------------+---------------+
|     8601868|      2153242|          25.03|
+------------+-------------+---------------+



In [54]:
# Validate Arrest completeness before calculating arrest rates
silver_df.groupBy("arrest").count().show()

+------+-------+
|arrest|  count|
+------+-------+
| false|6448626|
|  true|2153242|
+------+-------+



In [55]:
# ¿Cuál es la tasa de arrestos por tipo de delito?

gold_arrest_rate_by_primary_type = (
    silver_df
    .groupBy("primary_type")
    .agg(
        F.count("*").alias("total_crimes"),
        F.sum(
            F.when(F.col("arrest") == True, 1).otherwise(0)
        ).alias("total_arrests")
    )
    .withColumn(
        "arrest_rate_pct",
        F.round(
            F.col("total_arrests") / F.col("total_crimes") * 100,
            2
        )
    )
    .orderBy(F.col("arrest_rate_pct").desc())
)

gold_arrest_rate_by_primary_type.show(truncate=False)

+---------------------------------+------------+-------------+---------------+
|primary_type                     |total_crimes|total_arrests|arrest_rate_pct|
+---------------------------------+------------+-------------+---------------+
|DOMESTIC VIOLENCE                |1           |1            |100.0          |
|PROSTITUTION                     |70523       |70204        |99.55          |
|NARCOTICS                        |768683      |763413       |99.31          |
|GAMBLING                         |14674       |14566        |99.26          |
|LIQUOR LAW VIOLATION             |15528       |15370        |98.98          |
|PUBLIC INDECENCY                 |234         |228          |97.44          |
|CONCEALED CARRY LICENSE VIOLATION|1816        |1755         |96.64          |
|INTERFERENCE WITH PUBLIC OFFICER |20949       |19206        |91.68          |
|WEAPONS VIOLATION                |128552      |93490        |72.73          |
|OBSCENITY                        |998         |722 

La tasa global de arrestos es del **25,03 %** (2.153.242 de 8.601.868 delitos). Por tipo, destacan tasas muy altas en delitos donde la detención suele ser inmediata — **PROSTITUTION** (99,55 %) y **NARCOTICS** (99,31 %) — frente a tasas mucho más bajas en delitos contra la propiedad como **BATTERY** (21,63 %) o **ASSAULT** (20,04 %), donde identificar y localizar al autor es más difícil. *(Se excluye **DOMESTIC VIOLENCE**, con un único registro y 100 % de arrestos, por no ser representativo.)*

#### 📊 Resultados

Se ha implementado una arquitectura **Medallion con PySpark** sobre más de **8,6 millones de registros** de delitos de Chicago.

- 🥉 **Bronze:** se preservan los datos de origen con transformaciones mínimas y metadatos de ingesta y trazabilidad.

- 🥈 **Silver:** se normaliza el esquema, se realiza el tipado de variables temporales, se reconstruye y valida la información geoespacial, se aplican reglas de calidad y se eliminan registros duplicados, obteniendo un dataset limpio y reutilizable.

- 🥇 **Gold:** se generan datasets agregados orientados al análisis de la evolución temporal de los delitos, su distribución por distrito y tipología, así como KPIs relacionados con la tasa de arrestos.